In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [2]:
df = pd.read_csv('HealthSearchQA.csv', header=None)
df.columns = ['question']

In [3]:
df.head()

,question
0,Are benign brain tumors serious?
1,Are boils and carbuncles curable?
2,Are bone cysts serious?
3,Are cold sores a herpes virus?
4,Are dental abscesses serious?


In [4]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['clean_question'] = df['question'].apply(clean_text)

In [17]:
# N-gram Analysis (Phrases)
# Change ngram_range to (1,1) for single words or (2,2) for phrases
vec = CountVectorizer(stop_words='english', ngram_range=(2, 2), max_features=50)
counts = vec.fit_transform(df['clean_question'])
phrase_freq = pd.DataFrame(counts.toarray(), columns=vec.get_feature_names_out()).sum().sort_values(ascending=False)
print("Top Phrases:\n", phrase_freq.head(30))

Top Phrases:
 main cause            56
long does             47
life expectancy       42
look like             40
warning signs         40
feel like             31
does mean             29
common cause          25
best treatment        24
survival rate         24
left untreated        16
main causes           13
mental illness        13
pain feel             12
life threatening      12
long live             11
whats difference      11
signs symptoms        11
kidney disease        11
breast cancer         10
deficiency causes     10
fastest way            9
cancer spread          9
cancer curable         9
blood pressure         9
spread quickly         8
autoimmune disease     8
early signs            8
liver disease          8
polycystic kidney      7
dtype: int64


In [9]:
# Topic Modeling (LDA)
lda_vec = CountVectorizer(stop_words='english', max_df=0.95, min_df=2)
dtm = lda_vec.fit_transform(df['clean_question'])
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
lda_model.fit(dtm)

# Display Topics
words = lda_vec.get_feature_names_out()
for idx, topic in enumerate(lda_model.components_):
    top_words = [words[i] for i in topic.argsort()[-10:]]
    print(f"Topic {idx+1}: {', '.join(top_words)}")

Topic 1: symptom, long, mean, causes, common, stop, cancer, pain, does, types
Topic 2: happens, normal, treat, warning, rid, expectancy, cancer, life, causes, signs
Topic 3: happens, rate, signs, stages, blood, look, feel, does, like, causes
Topic 4: cured, disorders, called, disorder, caused, common, syndrome, main, symptoms, cause
Topic 5: curable, treatment, best, know, cancer, mean, long, away, disease, does


In [19]:
import os
import xml.etree.ElementTree as ET

def extract_focus_tags(directory_name, output_file):
    focus_list = []

    # Ensure the directory exists
    if not os.path.exists(directory_name):
        print(f"Directory '{directory_name}' not found.")
        return

    # Iterate through every file in the folder
    for filename in os.listdir(directory_name):
        if filename.endswith(".xml"):
            file_path = os.path.join(directory_name, filename)
            
            try:
                # Parse the XML file
                tree = ET.parse(file_path)
                root = tree.getroot()

                # Find all <Focus> tags (even if nested)
                for focus in root.iter('Focus'):
                    if focus.text:
                        focus_list.append(focus.text.strip())
            
            except ET.ParseError:
                print(f"Could not parse {filename}. Skipping...")

    # Save the results to a txt file
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in focus_list:
            f.write(f"{item}\n")

    print(f"Success! Extracted {len(focus_list)} phrases to {output_file}.")

# Run the function
extract_focus_tags('MPlusHerbsSupplements_QA', 'extracted_herbal_supplements.txt')

Success! Extracted 99 phrases to extracted_herbal_supplements.txt.
